In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, glob, re
from typing import Dict, List, Optional, Tuple, Set

import numpy as np
import pandas as pd
import torch
import dgl
import h5py

# =========================
# paths
# =========================
# dataset = "LGG"
# dataset = "BRCA"
dataset = "COAD"

WSI_ROOT = f"/data5/zhangye/morn/data/wsi/{dataset}"
OMICS_DIR = f"/data5/zhangye/morn/data/raw/{dataset}"

OMICS_FILES = {
    "CNV":   os.path.join(OMICS_DIR, f"{dataset}_CNV_top.csv"),
    "Methy": os.path.join(OMICS_DIR, f"{dataset}_Methy_top.csv"),
    "mRNA":  os.path.join(OMICS_DIR, f"{dataset}_mRNA_top.csv"),
    "miRNA": os.path.join(OMICS_DIR, f"{dataset}_miRNA_top.csv"),
}

GMT_PATH = "/data5/zhangye/morn/data/ReactomePathways.gmt"
MTI_PATH = "/data5/zhangye/morn/data/hsa_MTI.csv"

WSI_H5_ROOT = f"/data5/zhangye/trident/trident_processed_{dataset}/20x_256px_0px_overlap/features_uni_v1"

# ✅ 分类标签：patient_id + label (数字编码)
CLS_LABEL_CSV = f"/data5/zhangye/morn/data/cls_label/{dataset}_classification_label.csv"
CLS_PID_COL = "patient_id"
CLS_LABEL_COL = "label"

SPLIT_DIR = f"/data5/zhangye/morn/data/splits_5folds/tcga_{dataset}"

OUT_DIR = f"../data/processed/{dataset}_hgt_dataset"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# hyperparams
# =========================
TOPK_PATIENT_OMICS = 50
WSI_TOPK_PATCH = 4096

# ✅ 新增：WSI patch 数阈值（少于该值就丢弃整个 patient+wsi）
MIN_WSI_PATCHES = 100

HUB_PCT = 80.0
MAX_HUBS_PER_PATHWAY = 5
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

CNV_USE_ABS_MEAN = True
METHY_USE_ABS_MEAN = True

# =========================
# helpers
# =========================
def list_patients_from_wsi_dir(wsi_root: str) -> List[str]:
    pts = []
    for name in os.listdir(wsi_root):
        p = os.path.join(wsi_root, name)
        if os.path.isdir(p) and name.startswith("TCGA-"):
            pts.append(name)
    return sorted(list(set(pts)))

def normalize_omics_pid(pid: str) -> Optional[str]:
    # TCGA-like: A.B.C... -> A-B-C
    parts = str(pid).split(".")
    if len(parts) < 3 or len(parts) > 4:
        return None
    return ".".join(parts[:3]).replace(".", "-")

def read_omics_csv(path: str) -> pd.DataFrame:
    assert os.path.isfile(path), f"Not found: {path}"
    return pd.read_csv(path, index_col=0)

def _norm_gene(x: str) -> str:
    return str(x).strip().upper() if x is not None else ""

def _norm_mirna(x: str, drop_arm: bool = True) -> str:
    """
    Normalize miRNA names to a canonical form: 'hsa-mir-xxx...'
    Handles:
      - hsa.mir.110  -> hsa-mir-110
      - hsa-miR-222  -> hsa-mir-222
      - mir.181b.2   -> hsa-mir-181b-2
      - MIR-181B-2   -> hsa-mir-181b-2
      - optional drop -3p/-5p
    """
    if x is None:
        return ""

    t = str(x).strip()
    if t == "":
        return ""

    t = t.lower()
    t = t.replace("_", "-").replace(".", "-")

    t = t.replace("hsa-mirna-", "hsa-mir-")
    t = t.replace("hsa-mirna", "hsa-mir-")

    if t.startswith("hsa-mir") and not t.startswith("hsa-mir-"):
        t = "hsa-mir-" + t[len("hsa-mir"):].lstrip("-")

    if t.startswith("mirna-"):
        t = "hsa-mir-" + t[len("mirna-"):]
    elif t.startswith("mirna"):
        rest = t[len("mirna"):].lstrip("-")
        if rest:
            t = "hsa-mir-" + rest
    elif t.startswith("mir-"):
        t = "hsa-mir-" + t[len("mir-"):]
    elif t.startswith("mir"):
        rest = t[len("mir"):].lstrip("-")
        if rest:
            t = "hsa-mir-" + rest

    if t.startswith("mir-"):
        t = "hsa-" + t

    t = re.sub(r"-{2,}", "-", t).strip("-")

    if drop_arm:
        t = re.sub(r"-(3p|5p)$", "", t)

    if not (t.startswith("hsa-mir-") or t.startswith("hsa-let-")):
        return ""

    return t

def load_gmt(gmt_path: str) -> Dict[str, Set[str]]:
    assert os.path.isfile(gmt_path), f"Not found: {gmt_path}"
    pw = {}
    with open(gmt_path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name = parts[0].strip()
            genes = set(_norm_gene(g) for g in parts[2:] if str(g).strip())
            genes = {g for g in genes if g}
            if name and genes:
                pw[name] = genes
    return pw

def _read_first_2d_dataset_from_h5(h5_path: str) -> np.ndarray:
    with h5py.File(h5_path, "r") as f:
        for key in ["features", "feat", "embeddings", "patch_features"]:
            if key in f and hasattr(f[key], "shape") and len(f[key].shape) == 2:
                return f[key][:]
        def _visit(name, obj):
            if hasattr(obj, "shape") and len(obj.shape) == 2:
                raise RuntimeError(name)
        try:
            f.visititems(_visit)
        except RuntimeError as e:
            k = str(e)
            return f[k][:]
    raise ValueError(f"No 2D dataset found in {h5_path}")

def _get_first_2d_dataset_shape_from_h5(h5_path: str) -> Optional[Tuple[int, int]]:
    """
    只读取shape，不把特征整个读进内存，用于快速判断patch数。
    返回 (N, D) 或 None
    """
    try:
        with h5py.File(h5_path, "r") as f:
            for key in ["features", "feat", "embeddings", "patch_features"]:
                if key in f and hasattr(f[key], "shape") and len(f[key].shape) == 2:
                    sh = f[key].shape
                    return int(sh[0]), int(sh[1])

            found = {"shape": None}

            def _visit(name, obj):
                if hasattr(obj, "shape") and len(obj.shape) == 2:
                    found["shape"] = (int(obj.shape[0]), int(obj.shape[1]))
                    raise RuntimeError("FOUND")

            try:
                f.visititems(_visit)
            except RuntimeError:
                pass

            return found["shape"]
    except Exception:
        return None

def _find_h5_for_patient(pid: str) -> Optional[str]:
    cand = sorted(glob.glob(os.path.join(WSI_H5_ROOT, f"{pid}-*.h5")))
    return cand[0] if len(cand) > 0 else None

def filter_patients_by_wsi_patches(patients: List[str], min_patches: int) -> Tuple[List[str], Dict[str, int]]:
    """
    过滤：h5不存在 / 读不到2D特征 / patch数 < min_patches 的都丢掉。
    返回：kept_patients, reason_count
    """
    kept = []
    reason_count = {
        "missing_h5": 0,
        "no_2d_dataset_or_read_error": 0,
        "too_few_patches": 0,
        "kept": 0,
    }

    for pid in patients:
        h5_path = _find_h5_for_patient(pid)
        if h5_path is None:
            reason_count["missing_h5"] += 1
            continue

        sh = _get_first_2d_dataset_shape_from_h5(h5_path)
        if sh is None:
            reason_count["no_2d_dataset_or_read_error"] += 1
            continue

        n_patches = sh[0]
        if n_patches < min_patches:
            reason_count["too_few_patches"] += 1
            continue

        kept.append(pid)

    reason_count["kept"] = len(kept)
    return kept, reason_count

def wsi_topk_keep_patches(h5_path: str, topk: int) -> Tuple[np.ndarray, np.ndarray]:
    feats = _read_first_2d_dataset_from_h5(h5_path).astype(np.float32)  # (N,D)
    n, d = feats.shape
    k = min(topk, n)
    out = np.zeros((topk, d), dtype=np.float32)
    mask = np.zeros((topk,), dtype=np.bool_)

    if n == 0 or k == 0:
        return out, mask

    scores = np.linalg.norm(feats, axis=1)
    idx = np.argpartition(scores, -k)[-k:]
    out[:k] = feats[idx]
    mask[:k] = True
    return out, mask

# -------------------------
# classification label loader
# -------------------------
def _normalize_pid_like_tcga(x: str) -> str:
    s = str(x).strip()
    if s.startswith("TCGA-"):
        parts = s.split("-")
        if len(parts) >= 3:
            return "-".join(parts[:3])
    if "." in s:
        p = normalize_omics_pid(s)
        if p is not None:
            return p
    return s

def load_classification_labels(
    label_csv: str,
    patients: List[str],
    pid_col: str = "patient_id",
    label_col: str = "label",
    ignore_labels: Optional[Set[int]] = None,
) -> Tuple[List[str], torch.Tensor, Dict[str, int]]:
    """
    返回：
      kept_patients: 只保留有label且不在ignore_labels中的patients（并按输入patients顺序）
      y: (len(kept_patients),) long tensor，直接用csv里的数字编码（不做重映射）
      stats: 统计信息
    """
    assert os.path.isfile(label_csv), f"Not found: {label_csv}"
    df = pd.read_csv(label_csv)

    assert pid_col in df.columns, f"Label file missing '{pid_col}'. got={list(df.columns)}"
    assert label_col in df.columns, f"Label file missing '{label_col}'. got={list(df.columns)}"

    df = df[[pid_col, label_col]].copy()
    df[pid_col] = df[pid_col].astype(str).map(_normalize_pid_like_tcga)

    # label 强制转 int（允许字符串数字）
    df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
    df = df.dropna(subset=[pid_col, label_col])
    df[label_col] = df[label_col].astype(int)

    # ✅ 新增：忽略某些label（例如COAD的label=3）
    ignore_labels = set(ignore_labels or [])
    if len(ignore_labels) > 0:
        before = len(df)
        df = df[~df[label_col].isin(ignore_labels)].copy()
        after = len(df)
        print(f"[CLS] ignore_labels={sorted(list(ignore_labels))} | dropped_rows={before - after}")

    pid2label = dict(zip(df[pid_col].tolist(), df[label_col].tolist()))

    kept_patients = []
    y_list = []
    missing = 0
    dropped_by_ignore = 0

    for p in patients:
        if p not in pid2label:
            missing += 1
            continue
        lab = int(pid2label[p])
        if lab in ignore_labels:
            # 理论上不会发生（上面df已经过滤），这里保底
            dropped_by_ignore += 1
            continue
        kept_patients.append(p)
        y_list.append(lab)

    if len(kept_patients) == 0:
        raise RuntimeError(f"No patients matched labels in {label_csv}. Check pid normalization / columns / ignore_labels.")

    y = torch.tensor(y_list, dtype=torch.long)
    uniq = sorted(list(set(y_list)))

    stats = {
        "patients_in": len(patients),
        "patients_kept": len(kept_patients),
        "patients_missing_label_or_dropped": missing,
        "patients_dropped_by_ignore_labels": int(dropped_by_ignore),
        "ignore_labels": sorted(list(ignore_labels)),
        "num_classes_observed": int(len(uniq)),
        "unique_labels_observed": uniq,
        "label_min": int(min(y_list)) if len(y_list) else None,
        "label_max": int(max(y_list)) if len(y_list) else None,
    }
    return kept_patients, y, stats

def load_split_indices(split_csv: str, patients: List[str]) -> Dict[str, torch.Tensor]:
    df = pd.read_csv(split_csv)
    patient2idx = {p: i for i, p in enumerate(patients)}

    def _col(col: str) -> torch.Tensor:
        pids = df[col].dropna().astype(str).str.strip().tolist()
        pids = [_normalize_pid_like_tcga(x) for x in pids]
        kept = [patient2idx[p] for p in pids if p in patient2idx]
        return torch.tensor(kept, dtype=torch.int64)

    for c in ["train", "val", "test"]:
        assert c in df.columns, f"Split file missing column {c}: got {list(df.columns)}"
    return {"train_idx": _col("train"), "val_idx": _col("val"), "test_idx": _col("test")}

# =========================
# OMICS: build patient-gene edges + weights
# =========================
def build_patient_gene_edges(
    df_pat_x_feat_T: pd.DataFrame,
    patients_keep: List[str],
    topk: int,
    rank_use_abs: bool,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], np.ndarray]:
    df = df_pat_x_feat_T.copy()
    df["__pid__"] = [normalize_omics_pid(x) for x in df.index]
    df = df[df["__pid__"].notna()].set_index("__pid__", drop=True)

    feats = df.drop(columns=[c for c in df.columns if str(c).startswith("__")], errors="ignore")
    col_names = list(feats.columns)

    P = len(patients_keep)
    G = len(col_names)

    X = np.zeros((P, G), dtype=np.float32)
    pid2row = {pid: i for i, pid in enumerate(feats.index)}
    for i, pid in enumerate(patients_keep):
        if pid in pid2row:
            X[i] = feats.iloc[pid2row[pid]].to_numpy(dtype=np.float32)

    us, vs, ws = [], [], []
    for p_idx in range(P):
        row = X[p_idx]
        score = np.abs(row) if rank_use_abs else row
        k = min(topk, score.shape[0])
        if k <= 0:
            continue
        idx = np.argpartition(score, -k)[-k:]
        us.extend([p_idx] * len(idx))
        vs.extend(idx.astype(np.int64).tolist())
        ws.extend(row[idx].astype(np.float32).tolist())

    return np.array(us, np.int64), np.array(vs, np.int64), np.array(ws, np.float32), col_names, X

# =========================
# MTI: miRNA->mRNA edges + hub mRNA
# =========================
def load_mti_edges(
    mti_path: str,
    mir_cols: List[str],
    mrna_cols: List[str],
    drop_arm: bool = True,
) -> Tuple[np.ndarray, np.ndarray, Set[str]]:
    df = pd.read_csv(mti_path)
    assert "miRNA" in df.columns and "Target Gene" in df.columns, f"Bad MTI columns: {list(df.columns)}"

    mir_map = {}
    for i, m in enumerate(mir_cols):
        k = _norm_mirna(m, drop_arm=drop_arm)
        if k and k not in mir_map:
            mir_map[k] = i

    mrna_map = {}
    for i, g in enumerate(mrna_cols):
        k = _norm_gene(g)
        if k and k not in mrna_map:
            mrna_map[k] = i

    mir_s = df["miRNA"].astype(str).map(lambda x: _norm_mirna(x, drop_arm=drop_arm))
    gene_s = df["Target Gene"].astype(str).map(_norm_gene)

    # ---- DEBUG overlap
    mti_mirs = set([x for x in mir_s.values.tolist() if x])
    col_mirs = set(mir_map.keys())
    overlap_mir = len(mti_mirs & col_mirs)

    mti_genes = set([x for x in gene_s.values.tolist() if x])
    col_genes = set(mrna_map.keys())
    overlap_gene = len(mti_genes & col_genes)

    print(f"[MTI-DEBUG] mir keys: cols={len(col_mirs)} mti={len(mti_mirs)} overlap={overlap_mir}")
    print(f"[MTI-DEBUG] gene keys: cols={len(col_genes)} mti={len(mti_genes)} overlap={overlap_gene}")
    if overlap_mir == 0:
        print("[MTI-DEBUG] example normalized mir from cols:", list(sorted(list(col_mirs)))[0:10])
        print("[MTI-DEBUG] example normalized mir from MTI :", list(sorted(list(mti_mirs)))[0:10])

    U, V = [], []
    tgt_counts: Dict[str, int] = {}
    for m, g in zip(mir_s, gene_s):
        if g in mrna_map:
            tgt_counts[g] = tgt_counts.get(g, 0) + 1
        if (m in mir_map) and (g in mrna_map):
            U.append(mir_map[m])
            V.append(mrna_map[g])

    hub_genes: Set[str] = set()
    if len(tgt_counts) > 0:
        vals = np.array(list(tgt_counts.values()), dtype=np.float32)
        thr = np.percentile(vals, HUB_PCT)
        hub_genes = {g for g, c in tgt_counts.items() if c >= thr}

    return np.array(U, np.int64), np.array(V, np.int64), hub_genes

def add_pathway_hub_edges(
    data_dict: dict,
    cnv_cols: List[str],
    met_cols: List[str],
    mrna_cols: List[str],
    pathway2genes: Dict[str, Set[str]],
    hub_mrna_genes: Set[str],
    max_hubs_per_pathway: int = 5,
):
    cnv_map   = {_norm_gene(g): i for i, g in enumerate(cnv_cols)}
    met_map   = {_norm_gene(g): i for i, g in enumerate(met_cols)}
    mrna_map  = {_norm_gene(g): i for i, g in enumerate(mrna_cols)}
    hub_norm  = {_norm_gene(g) for g in hub_mrna_genes}

    U_cnv, V_cnv = [], []
    U_met, V_met = [], []

    for _, genes in pathway2genes.items():
        hubs = [g for g in genes if (g in hub_norm) and (g in mrna_map)]
        if len(hubs) == 0:
            continue
        hubs = hubs[:max_hubs_per_pathway]
        hub_ids = [mrna_map[g] for g in hubs]

        for g in genes:
            if g in cnv_map:
                u = cnv_map[g]
                for v in hub_ids:
                    U_cnv.append(u); V_cnv.append(v)
            if g in met_map:
                u = met_map[g]
                for v in hub_ids:
                    U_met.append(u); V_met.append(v)

    if len(U_cnv) > 0:
        U = np.array(U_cnv, np.int64); V = np.array(V_cnv, np.int64)
        data_dict[("gene_CNV", "cnv_to_mrna", "gene_mRNA")] = (U, V)
        data_dict[("gene_mRNA", "mrna_to_cnv", "gene_CNV")] = (V, U)

    if len(U_met) > 0:
        U = np.array(U_met, np.int64); V = np.array(V_met, np.int64)
        data_dict[("gene_Methy", "methy_to_mrna", "gene_mRNA")] = (U, V)
        data_dict[("gene_mRNA", "mrna_to_methy", "gene_Methy")] = (V, U)

# =========================
# Main
# =========================
patients_all = list_patients_from_wsi_dir(WSI_ROOT)
print(f"[WSI] {dataset} patients (raw from dir) = {len(patients_all)}")

# 1) WSI patch 过滤
patients_wsi, rc = filter_patients_by_wsi_patches(patients_all, min_patches=MIN_WSI_PATCHES)
print(f"[WSI-FILTER] MIN_WSI_PATCHES={MIN_WSI_PATCHES} | kept={rc['kept']} | "
      f"missing_h5={rc['missing_h5']} | read_err={rc['no_2d_dataset_or_read_error']} | too_few={rc['too_few_patches']}")

if len(patients_wsi) == 0:
    raise RuntimeError("After filtering by WSI patches, no patients left. Check WSI_H5_ROOT / MIN_WSI_PATCHES.")

# 2) 分类 label 过滤（没有label的病人直接跳过）
# ✅ COAD 特例：忽略 label=3（第4类样本太少）
ignore_labels = {3} if dataset == "COAD" else set()

patients, cls_y, cls_stats = load_classification_labels(
    CLS_LABEL_CSV,
    patients_wsi,
    pid_col=CLS_PID_COL,
    label_col=CLS_LABEL_COL,
    ignore_labels=ignore_labels,
)

print(f"[CLS] label_csv={CLS_LABEL_CSV}")
print(f"[CLS] kept={cls_stats['patients_kept']} | missing_or_dropped={cls_stats['patients_missing_label_or_dropped']} | "
      f"ignore_labels={cls_stats['ignore_labels']} | classes_observed={cls_stats['num_classes_observed']} | "
      f"unique={cls_stats['unique_labels_observed']} | label_range=[{cls_stats['label_min']},{cls_stats['label_max']}]")

P = len(patients)
num_nodes_dict = {"patient": P, "wsi": P}

data_dict = {}
edge_w = {}  # canonical etype -> weight array (float32)

# ---- patient <-> wsi (weight=1)
u = np.arange(P, dtype=np.int64)
v = np.arange(P, dtype=np.int64)
data_dict[("patient", "has_wsi", "wsi")] = (u, v)
data_dict[("wsi", "belongs_to", "patient")] = (v, u)
edge_w[("patient", "has_wsi", "wsi")] = np.ones((P,), np.float32)
edge_w[("wsi", "belongs_to", "patient")] = np.ones((P,), np.float32)

# ---- patient <-> omics
omics_cols = {}
omics_X = {}

for om, fpath in OMICS_FILES.items():
    df = read_omics_csv(fpath).T
    rank_use_abs = True if om in ["CNV", "Methy"] else False

    pu, gv, w_val, cols, X = build_patient_gene_edges(df, patients, TOPK_PATIENT_OMICS, rank_use_abs=rank_use_abs)

    if om == "CNV":
        ntype, e_fwd, e_rev = "gene_CNV", "has_cnv", "in_patient"
    elif om == "Methy":
        ntype, e_fwd, e_rev = "gene_Methy", "has_methy", "in_patient"
    elif om == "mRNA":
        ntype, e_fwd, e_rev = "gene_mRNA", "has_mrna", "in_patient"
    elif om == "miRNA":
        ntype, e_fwd, e_rev = "gene_miRNA", "has_mirna", "in_patient"
    else:
        raise ValueError(om)

    omics_cols[ntype] = cols
    omics_X[ntype] = X
    num_nodes_dict[ntype] = len(cols)

    data_dict[("patient", e_fwd, ntype)] = (pu, gv)
    data_dict[(ntype, e_rev, "patient")] = (gv, pu)

    # edge weight rule
    if ntype == "gene_mRNA":
        edge_w[("patient", e_fwd, ntype)] = w_val.astype(np.float32)
        edge_w[(ntype, e_rev, "patient")] = w_val.astype(np.float32)
    else:
        edge_w[("patient", e_fwd, ntype)] = np.ones((len(pu),), np.float32)
        edge_w[(ntype, e_rev, "patient")] = np.ones((len(pu),), np.float32)

    print(f"[OMICS] {om}: nodes={len(cols)} | edges={len(pu)}")

# ---- miRNA -> mRNA edges (MTI), weight = miRNA expression mean
print("[MTI] Building miRNA->mRNA edges...")
U_mti, V_mti, hub_mrna_genes = load_mti_edges(
    MTI_PATH,
    mir_cols=omics_cols["gene_miRNA"],
    mrna_cols=omics_cols["gene_mRNA"],
    drop_arm=True,
)

if len(U_mti) > 0:
    data_dict[("gene_miRNA", "targets", "gene_mRNA")] = (U_mti, V_mti)
    data_dict[("gene_mRNA", "targeted_by", "gene_miRNA")] = (V_mti, U_mti)

    X_mir = omics_X["gene_miRNA"]
    mir_mean = X_mir.mean(axis=0).astype(np.float32)
    w_mti = mir_mean[U_mti].astype(np.float32)

    edge_w[("gene_miRNA", "targets", "gene_mRNA")] = w_mti
    edge_w[("gene_mRNA", "targeted_by", "gene_miRNA")] = w_mti
    print(f"[MTI] edges={len(U_mti)} (weight=miRNA mean expression)")
else:
    print("[WARN] MTI edges = 0 (name mismatch?)")

# ---- pathway hub edges: CNV/Methy -> hub mRNA
print("[GMT] Loading Reactome GMT + building CNV/Methy -> hub mRNA edges...")
pathway2genes = load_gmt(GMT_PATH)
add_pathway_hub_edges(
    data_dict,
    cnv_cols=omics_cols["gene_CNV"],
    met_cols=omics_cols["gene_Methy"],
    mrna_cols=omics_cols["gene_mRNA"],
    pathway2genes=pathway2genes,
    hub_mrna_genes=hub_mrna_genes,
    max_hubs_per_pathway=MAX_HUBS_PER_PATHWAY,
)

if ("gene_Methy", "methy_to_mrna", "gene_mRNA") in data_dict:
    U, V = data_dict[("gene_Methy", "methy_to_mrna", "gene_mRNA")]
    X_met = omics_X["gene_Methy"]
    met_mean = (np.abs(X_met).mean(axis=0) if METHY_USE_ABS_MEAN else X_met.mean(axis=0)).astype(np.float32)
    w = met_mean[U].astype(np.float32)
    edge_w[("gene_Methy", "methy_to_mrna", "gene_mRNA")] = w
    edge_w[("gene_mRNA", "mrna_to_methy", "gene_Methy")] = w
    print(f"[PATH-HUB] Methy->mRNA edges={len(U)} (weight=methy mean)")

if ("gene_CNV", "cnv_to_mrna", "gene_mRNA") in data_dict:
    U, V = data_dict[("gene_CNV", "cnv_to_mrna", "gene_mRNA")]
    X_cnv = omics_X["gene_CNV"]
    cnv_mean = (np.abs(X_cnv).mean(axis=0) if CNV_USE_ABS_MEAN else X_cnv.mean(axis=0)).astype(np.float32)
    w = cnv_mean[U].astype(np.float32)
    edge_w[("gene_CNV", "cnv_to_mrna", "gene_mRNA")] = w
    edge_w[("gene_mRNA", "mrna_to_cnv", "gene_CNV")] = w
    print(f"[PATH-HUB] CNV->mRNA edges={len(U)} (weight=cnv mean)")

# ---- build graph
g = dgl.heterograph(data_dict, num_nodes_dict=num_nodes_dict)
print("[GRAPH]", g)

# attach edge weights (missing -> 1)
for et in g.canonical_etypes:
    n_e = g.num_edges(et)
    if et in edge_w:
        w = edge_w[et]
        assert len(w) == n_e, f"edge weight length mismatch for {et}: {len(w)} vs {n_e}"
        g.edges[et].data["w"] = torch.tensor(w, dtype=torch.float32)
    else:
        g.edges[et].data["w"] = torch.ones((n_e,), dtype=torch.float32)

# node ids
for ntype in g.ntypes:
    g.nodes[ntype].data["nid"] = torch.arange(g.num_nodes(ntype), dtype=torch.long)

# ---- WSI patch feats (K=4096)
print(f"[WSI-PATCH] Saving topK patches per WSI: K={WSI_TOPK_PATCH} (patients kept={P})")
feat_dim = None

wsi_patches_list = []
wsi_mask_list = []

for pid in patients:
    h5_path = _find_h5_for_patient(pid)
    if h5_path is None:
        # 理论上不会缺失（前面wsi过滤过），但保底
        wsi_patches_list.append(None)
        wsi_mask_list.append(None)
        continue
    patches, mask = wsi_topk_keep_patches(h5_path, topk=WSI_TOPK_PATCH)
    feat_dim = feat_dim or int(patches.shape[1])
    wsi_patches_list.append(patches)
    wsi_mask_list.append(mask)

if feat_dim is None:
    raise RuntimeError("No WSI h5 features found. Check WSI_H5_ROOT.")

wsi_patches = np.zeros((P, WSI_TOPK_PATCH, feat_dim), dtype=np.float32)
wsi_mask    = np.zeros((P, WSI_TOPK_PATCH), dtype=np.bool_)
for i in range(P):
    if wsi_patches_list[i] is None:
        continue
    wsi_patches[i] = wsi_patches_list[i]
    wsi_mask[i]    = wsi_mask_list[i]

g.nodes["wsi"].data["wsi_patches"] = torch.tensor(wsi_patches, dtype=torch.float32)
g.nodes["wsi"].data["wsi_patch_mask"] = torch.tensor(wsi_mask, dtype=torch.bool)

# ---- classification labels (no mapping)
g.nodes["patient"].data["label"] = cls_y
print("[LABEL] labeled patients =", int(len(cls_y)))
print("[LABEL] unique labels =", sorted(list(set(cls_y.tolist()))))

# ---- save graph
graph_path = os.path.join(OUT_DIR, f"{dataset}_graph.bin")
dgl.save_graphs(graph_path, [g])
print(f"[SAVED] {graph_path}")

# ---- save each fold split/meta
split_csvs = sorted(glob.glob(os.path.join(SPLIT_DIR, "splits_*.csv")))
if len(split_csvs) == 0:
    split_csvs = sorted(glob.glob(os.path.join(SPLIT_DIR, "*.csv")))
assert len(split_csvs) > 0, f"No split csv found under: {SPLIT_DIR}"
print(f"[SPLIT] found {len(split_csvs)} split files.")

for split_csv in split_csvs:
    fold_name = os.path.splitext(os.path.basename(split_csv))[0]
    fold_out = os.path.join(OUT_DIR, fold_name)
    os.makedirs(fold_out, exist_ok=True)

    splits = load_split_indices(split_csv, patients)
    train_idx = splits["train_idx"]
    val_idx   = splits["val_idx"]
    test_idx  = splits["test_idx"]

    torch.save(
        {"train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx},
        os.path.join(fold_out, f"{fold_name}_split.pt")
    )

    meta = {
        "dataset": dataset,
        "patients_raw_before_wsi_filter": patients_all,
        "patients_after_wsi_filter": patients_wsi,
        "patients_after_cls_filter": patients,
        "patients": patients,

        "wsi_filter": {
            "min_wsi_patches": MIN_WSI_PATCHES,
            "kept": rc["kept"],
            "missing_h5": rc["missing_h5"],
            "no_2d_dataset_or_read_error": rc["no_2d_dataset_or_read_error"],
            "too_few_patches": rc["too_few_patches"],
        },

        "cls_label": {
            "label_csv": CLS_LABEL_CSV,
            "pid_col": CLS_PID_COL,
            "label_col": CLS_LABEL_COL,
            "stats": cls_stats,
            "unique_labels": sorted(list(set(cls_y.tolist()))),
        },

        "num_nodes": {k: int(v) for k, v in num_nodes_dict.items()},
        "canonical_etypes": [list(x) for x in g.canonical_etypes],
        "topk_patient_omics": TOPK_PATIENT_OMICS,
        "wsi_topk_patch": WSI_TOPK_PATCH,
        "wsi_feat_dim": int(feat_dim),

        "split_csv": split_csv,
        "graph_bin": graph_path,

        "edge_weight_rule": {
            "patient<->mRNA": "mRNA expression value per (patient,gene) edge",
            "miRNA->mRNA": "mean miRNA expression (over patients) on each edge",
            "Methy->mRNA": "mean Methy value (over patients) on each edge",
            "CNV->mRNA": "mean CNV value (over patients, abs_mean by default) on each edge",
            "others": "1",
        },

        "split_sizes_after_filter": {
            "train": int(len(train_idx)),
            "val": int(len(val_idx)),
            "test": int(len(test_idx)),
        }
    }

    with open(os.path.join(fold_out, f"{fold_name}_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[FOLD {fold_name}] train/val/test = {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")

print("[DONE]")

/home/zhangye/anaconda3/envs/morn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[WSI] COAD patients (raw from dir) = 285
[WSI-FILTER] MIN_WSI_PATCHES=100 | kept=271 | missing_h5=14 | read_err=0 | too_few=0
[CLS] ignore_labels=[3] | dropped_rows=4
[CLS] label_csv=/data5/zhangye/morn/data/cls_label/COAD_classification_label.csv
[CLS] kept=246 | missing_or_dropped=25 | ignore_labels=[3] | classes_observed=3 | unique=[0, 1, 2] | label_range=[0,2]
[OMICS] CNV: nodes=5000 | edges=12300
[OMICS] Methy: nodes=5000 | edges=12300
[OMICS] mRNA: nodes=5000 | edges=12300
[OMICS] miRNA: nodes=200 | edges=12300
[MTI] Building miRNA->mRNA edges...
[MTI-DEBUG] mir keys: cols=200 mti=1803 overlap=180
[MTI-DEBUG] gene keys: cols=5000 mti=16981 overlap=4140
[MTI] edges=147907 (weight=miRNA mean expression)
[GMT] Loading Reactome GMT + building CNV/Methy -> hub mRNA edges...
[PATH-HUB] Methy->mRNA edges=127264 (weight=methy mean)
[PATH-HUB] CNV->mRNA edges=126443 (weight=cnv mean)
[GRAPH] Graph(num_nodes={'gene_CNV': 5000, 'gene_Methy': 5000, 'gene_mRNA': 5000, 'gene_miRNA': 200, 'pati

In [4]:
import os
import numpy as np
import torch
import dgl

GRAPH_BIN = "../data/processed/LGG_hgt_dataset/LGG_graph.bin"  # <- 改成你的实际路径

graphs, _ = dgl.load_graphs(GRAPH_BIN)
g = graphs[0]

assert "wsi" in g.ntypes, "Graph has no 'wsi' ntype"
assert "wsi_patches" in g.nodes["wsi"].data, "Missing wsi_patches"
assert "wsi_patch_mask" in g.nodes["wsi"].data, "Missing wsi_patch_mask"

wsi_patches = g.nodes["wsi"].data["wsi_patches"]          # (P, 4096, D)
wsi_mask    = g.nodes["wsi"].data["wsi_patch_mask"]       # (P, 4096) bool
P, K, D = wsi_patches.shape

print(f"[WSI] shape = (P={P}, K={K}, D={D})")

# -----------------------
# 1) 每个病人有效patch数量
# -----------------------
valid_counts = wsi_mask.sum(dim=1).cpu().numpy()  # (P,)
print(f"[COUNT] min/mean/max valid patches = {valid_counts.min()} / {valid_counts.mean():.1f} / {valid_counts.max()}")

# 哪些 <4096
idx_lt = np.where(valid_counts < K)[0]
print(f"[COUNT] patients with <{K} valid patches: {len(idx_lt)}/{P}")

# 缺失h5：mask全False
idx_missing = np.where(valid_counts == 0)[0]
print(f"[MISSING H5] patients with 0 valid patches: {len(idx_missing)}/{P}")

# -----------------------
# 2) 不足4096时如何处理（检查 padded 部分是否为0）
# -----------------------
# padded 部分应该满足：mask=False 且 feature==0
# 检查：对每个病人，padded区域的绝对值最大值
patches_np = wsi_patches.cpu()
mask_np = wsi_mask.cpu()

# padded positions mask==False
padded_max_abs = []
padded_has_nonzero = 0
for i in range(P):
    pad_pos = ~mask_np[i]
    if pad_pos.any():
        mx = patches_np[i][pad_pos].abs().max().item()
        padded_max_abs.append(mx)
        if mx > 1e-6:
            padded_has_nonzero += 1
    else:
        padded_max_abs.append(0.0)

padded_max_abs = np.array(padded_max_abs)
print(f"[PADDED] padded max|x|: min/mean/max = {padded_max_abs.min():.3e} / {padded_max_abs.mean():.3e} / {padded_max_abs.max():.3e}")
print(f"[PADDED] patients whose padded region has non-zero values (>1e-6): {padded_has_nonzero}/{P}")

# -----------------------
# 3) 数值健康检查：NaN/Inf、以及 mask=True 但全0 的异常
# -----------------------
isfinite_all = torch.isfinite(wsi_patches).all().item()
print(f"[FINITE] wsi_patches all finite? {isfinite_all}")

nan_cnt = torch.isnan(wsi_patches).sum().item()
inf_cnt = torch.isinf(wsi_patches).sum().item()
print(f"[FINITE] NaN count = {nan_cnt}, Inf count = {inf_cnt}")

# mask=True 但 patch 全0（可能是读错/写错/某些h5为空但mask误置）
# 判定：在 mask=True 的位置，L2 norm == 0
valid_feats = wsi_patches[mask_np]
if valid_feats.numel() > 0:
    valid_norm0 = (valid_feats.abs().sum(dim=1) == 0).sum().item()  # sum over D
    print(f"[VALID-ZERO] valid patches that are exactly all-zero = {valid_norm0} (out of {valid_feats.shape[0]})")
else:
    print("[VALID-ZERO] no valid patches at all (all masks are False)")

# -----------------------
# 4) 输出前20个 “有效patch最少” 的病人索引，便于你定位
# -----------------------
order = np.argsort(valid_counts)
print("[TOP FEW] 20 patients with fewest valid patches (idx, count):")
for i in order[:20]:
    print(f"  {i:4d}  {valid_counts[i]}")

[WSI] shape = (P=245, K=4096, D=1024)
[COUNT] min/mean/max valid patches = 249 / 3010.9 / 4096
[COUNT] patients with <4096 valid patches: 116/245
[MISSING H5] patients with 0 valid patches: 0/245


/tmp/ipykernel_4707/1011937691.py:49: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at ../aten/src/ATen/native/IndexingUtils.h:27.)
  mx = patches_np[i][pad_pos].abs().max().item()


[PADDED] padded max|x|: min/mean/max = 5.723e+00 / 6.767e+00 / 8.721e+00
[PADDED] patients whose padded region has non-zero values (>1e-6): 245/245
[FINITE] wsi_patches all finite? True
[FINITE] NaN count = 0, Inf count = 0


/tmp/ipykernel_4707/1011937691.py:72: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at ../aten/src/ATen/native/IndexingUtils.h:27.)
  valid_feats = wsi_patches[mask_np]


[VALID-ZERO] valid patches that are exactly all-zero = 0 (out of 737669)
[TOP FEW] 20 patients with fewest valid patches (idx, count):
   147  249
   219  262
   124  315
   212  324
   210  377
   122  400
   141  405
   237  474
   208  535
   120  593
   121  641
   123  678
   106  683
   136  718
   125  743
   158  744
   218  745
   145  750
   214  750
   129  757


In [10]:
import numpy as np
import torch
import dgl

GRAPH_BIN = "../data/processed/COAD_hgt_dataset/COAD_graph.bin"  # 改成你的路径

g = dgl.load_graphs(GRAPH_BIN)[0][0]
x = g.nodes["wsi"].data["wsi_patches"].cpu()        # (P,K,D)
m = g.nodes["wsi"].data["wsi_patch_mask"].cpu()     # 可能是 uint8 / bool

# ✅ 强制转 bool（关键）
m = m.bool()

P, K, D = x.shape
valid_counts = m.sum(dim=1).numpy()

# 计算 padding 区域最大绝对值
padded_max = np.zeros(P, dtype=np.float32)
for i in range(P):
    pad_pos = (~m[i])              # bool 的反转才是对的
    if pad_pos.any():
        padded_max[i] = x[i][pad_pos].abs().max().item()
    else:
        padded_max[i] = 0.0

print("[MASK dtype]", m.dtype)
print("[COUNT] min/mean/max valid =", valid_counts.min(), valid_counts.mean(), valid_counts.max())
print("[PADDED max|x|] min/mean/max =", padded_max.min(), padded_max.mean(), padded_max.max())
print("[PADDED nonzero >1e-6] =", int((padded_max > 1e-6).sum()), "/", P)

# 专门看缺失 h5 的病人（valid=0）
idx_missing = np.where(valid_counts == 0)[0]
print("[MISSING] n =", len(idx_missing))
if len(idx_missing) > 0:
    print("Example missing idx:", idx_missing[:10].tolist())
    # 看看这些人的 x 是否全0
    mx = x[idx_missing].abs().max().item()
    print("[MISSING] max|x| over missing =", mx)

[MASK dtype] torch.bool
[COUNT] min/mean/max valid = 0 3628.2491228070176 4096
[PADDED max|x|] min/mean/max = 0.0 0.0 0.0
[PADDED nonzero >1e-6] = 0 / 285
[MISSING] n = 14
Example missing idx: [50, 51, 52, 53, 54, 55, 57, 58, 59, 60]
[MISSING] max|x| over missing = 0.0


In [4]:
import pandas as pd

df_label = pd.read_csv("/data5/zhangye/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-LGG/Aligned/54814574_LGG_label_num.csv")          # 247 x 1
df_cnv   = pd.read_csv("/data5/zhangye/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-LGG/Aligned/LGG_CNV_aligned.csv")                # 11205 x 248

sample_ids = [c for c in df_cnv.columns if c != "Unnamed: 0"]
assert len(sample_ids) == len(df_label)

def to_patient(s):
    # TCGA.CS.4938.01 -> TCGA-CS-4938
    p = s.split(".")
    return "-".join(p[:3]) if len(p) >= 3 and p[0].upper()=="TCGA" else s

map_df = pd.DataFrame({
    "sample_id": sample_ids,
    "patient_id": [to_patient(s) for s in sample_ids],
    "label": df_label["Label"].astype(int).values
})
map_df.to_csv("/data5/zhangye/morn/data/cls_label/LGG_classification_label.csv", index=False)

In [6]:
import os
import torch

def build_edge_groups_for_selector(G_full):
    eg = {
        "wsi": [],
        "patient_omics": {"CNV": [], "Methy": [], "mRNA": [], "miRNA": []},
        "regulation": {"mti": [], "pathway_hub": []},
    }

    for (s, e, d) in G_full.canonical_etypes:
        # -----------------------
        # WSI-related edges
        # -----------------------
        if s == "wsi" or d == "wsi":
            eg["wsi"].append((s, e, d))
            continue

        # -----------------------
        # patient <-> omics edges
        # -----------------------
        if s == "patient" or d == "patient":
            other = d if s == "patient" else s

            # 根据你的 ntype 命名改这里：gene_CNV / gene_Methy / gene_mRNA / gene_miRNA
            if other == "gene_CNV":
                eg["patient_omics"]["CNV"].append((s, e, d))
                continue
            if other == "gene_Methy":
                eg["patient_omics"]["Methy"].append((s, e, d))
                continue
            if other == "gene_mRNA":
                eg["patient_omics"]["mRNA"].append((s, e, d))
                continue
            if other == "gene_miRNA":
                eg["patient_omics"]["miRNA"].append((s, e, d))
                continue

        # -----------------------
        # regulation edges
        # -----------------------
        # mti: miRNA -> mRNA (或你实际定义的 miRNA target 关系)
        if (s == "gene_miRNA" and d == "gene_mRNA") or (s == "gene_mRNA" and d == "gene_miRNA"):
            eg["regulation"]["mti"].append((s, e, d))
            continue

        # pathway_hub: pathway <-> gene_* 或 pathway <-> patient (按你项目实际命名)
        # 你如果没有 pathway 这个 ntype，就把这段删掉或改成你的 hub 节点类型
        if s == "pathway" or d == "pathway":
            eg["regulation"]["pathway_hub"].append((s, e, d))
            continue

    return eg

def ensure_edge_groups_pt_compatible(G_full, data_dir):
    p = os.path.join(data_dir, "edge_groups.pt")
    eg = build_edge_groups_for_selector(G_full)
    torch.save(eg, p)
    print(f"[ABL] Wrote selector-compatible edge_groups.pt -> {p}")
    print("[ABL] counts:",
          "wsi=", len(eg["wsi"]),
          "patient_CNV=", len(eg["patient_omics"]["CNV"]),
          "patient_Methy=", len(eg["patient_omics"]["Methy"]),
          "patient_mRNA=", len(eg["patient_omics"]["mRNA"]),
          "patient_miRNA=", len(eg["patient_omics"]["miRNA"]),
          "mti=", len(eg["regulation"]["mti"]),
          "pathway_hub=", len(eg["regulation"]["pathway_hub"]))
    return p

In [8]:
import dgl
graphs, _ = dgl.load_graphs('/data5/zhangye/morn/data/processed/KIRP_hgt_dataset/KIRP_graph.bin')
G_full = graphs[0]
ensure_edge_groups_file(G_full, '/data5/zhangye/morn/data/processed/KIRP_hgt_dataset')

[ABL] Auto-created edge_groups.pt -> /data5/zhangye/morn/data/processed/KIRP_hgt_dataset/edge_groups.pt
[ABL] Groups:
  - omics_omics: 6 etypes
  - patient_omics: 8 etypes
  - patient_wsi: 2 etypes
  - patient_gene_CNV: 2 etypes
  - patient_gene_Methy: 2 etypes
  - patient_gene_mRNA: 2 etypes
  - patient_gene_miRNA: 2 etypes


'/data5/zhangye/morn/data/processed/KIRP_hgt_dataset/edge_groups.pt'